# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# UNI patch attribution for colorectal histology

**Research question:** Do pathology-pretrained Transformer attribution maps faithfully identify the image patches that drive colorectal histology classification?

This workflow keeps classification performance separate from explanation faithfulness. It reuses the prior case-grouped split, trains a lightweight classifier on a frozen UNI encoder, verifies one seed first, and then evaluates 14 x 14 patch explanations using patch occlusion and deletion tests.

The maps are prediction-contributing regions under a particular model and attribution method. They are not biological causes, tumor boundaries, or pathological annotations.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch.utils.data import DataLoader

sns.set_theme(style="whitegrid")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
DATASET_DIR = (
    PROJECT_ROOT
    / "Colorectal Histology MNIST"
    / "Kather_texture_2016_image_tiles_5000"
    / "Kather_texture_2016_image_tiles_5000"
)
BASELINE_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "baseline_cnn"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "uni_attribution"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
FEATURE_DIR = ARTIFACT_DIR / "features"
TOKEN_DIR = ARTIFACT_DIR / "patch_tokens"
PLOT_DIR = ARTIFACT_DIR / "plots"
for directory in (ARTIFACT_DIR, CHECKPOINT_DIR, FEATURE_DIR, TOKEN_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)
print("Dataset:", DATASET_DIR)
print("Artifacts:", ARTIFACT_DIR)

In [ ]:
from Methods.BaselineCNN import BaselineCNN, set_seed
from Methods.UNIAttribution import (
    UNI_GRID_SIZE,
    UNIImageDataset,
    build_feature_loaders,
    build_image_loaders,
    build_uni_classifier,
    cache_patch_tokens,
    classification_row,
    evaluate_cnn_examples,
    evaluate_frozen_head,
    evaluate_uni_examples,
    evaluate_uni_seed_stability,
    extract_cls_feature_cache,
    fit_frozen_head,
    load_classifier_head,
    load_fixed_manifest,
    load_normalized_image,
    resolve_uni_transform,
    select_representative_examples,
)

SEED = 41
MULTI_SEEDS = (11, 23, 41, 57, 73, 89, 101, 131, 151, 181)
NUM_WORKERS = 4
IMAGE_BATCH_SIZE = 24
FEATURE_BATCH_SIZE = 128
HEAD_EPOCHS = 40
HEAD_LEARNING_RATE = 1e-3
FORCE_FEATURE_REBUILD = False
UNI_MODEL_NAME = "hf-hub:MahmoodLab/uni"
LOGIN_TO_HUGGINGFACE = True
HF_TOKEN = os.environ.get("HF_TOKEN")  # Leave unset to use the secure notebook prompt.
LOCAL_UNI_ASSETS_DIR = None  # Optional fallback: PROJECT_ROOT / "uni" / "assets" / "ckpts"
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
if LOGIN_TO_HUGGINGFACE:
    from huggingface_hub import login, whoami

    login(token=HF_TOKEN, add_to_git_credential=False)
    hf_account = whoami()
    print("Hugging Face account:", hf_account.get("name", hf_account))
set_seed(SEED)
DEVICE

## 1. Reuse the fixed case-grouped split

The previous manifest is joined by relative image path so the UNI experiment uses exactly the earlier train, validation, and test assignments. If the artifact is unavailable, the same seed-41 grouped split is recreated.

In [ ]:
previous_manifest_candidates = [
    BASELINE_ARTIFACT_DIR / "split_manifest.csv",
    BASELINE_ARTIFACT_DIR / "split_manifest_without_masks.csv",
]
previous_manifest_path = next(
    (path for path in previous_manifest_candidates if path.is_file()),
    previous_manifest_candidates[0],
)
manifest = load_fixed_manifest(
    DATASET_DIR,
    previous_manifest_path=previous_manifest_path,
    random_state=SEED,
)
manifest.to_csv(ARTIFACT_DIR / "split_manifest.csv", index=False)

class_table = manifest[["class_name", "label"]].drop_duplicates().sort_values("label")
CLASS_NAMES = class_table["class_name"].tolist()
print("Images:", len(manifest))
print("Cases:", sorted(manifest["case_id"].unique()))
display(pd.crosstab(manifest["split"], manifest["class_name"]))
display(manifest.groupby("split")["case_id"].unique().to_frame())

## 2. Load frozen UNI and cache CLS features

UNI is loaded through timm from the gated `hf-hub:MahmoodLab/uni` repository. Before running this section, accept the model's access conditions on Hugging Face and run the configuration cell so `login()` authenticates this notebook environment. Its evaluation transform is constructed directly from `model.pretrained_cfg` with `resolve_data_config()` and `create_transform()`, ensuring that feature extraction and attribution use the checkpoint's official preprocessing. The deterministic frozen CLS features are cached once, while patch tokens remain available from `forward_features()` for explanation.

In [ ]:
uni_model = build_uni_classifier(
    PROJECT_ROOT,
    num_classes=len(CLASS_NAMES),
    device=DEVICE,
    model_name=UNI_MODEL_NAME,
    assets_dir=LOCAL_UNI_ASSETS_DIR,
)
uni_transform, uni_data_config = resolve_uni_transform(uni_model.encoder)
UNI_PREPROCESSING_ID = f"{UNI_MODEL_NAME}|{repr(sorted(uni_data_config.items()))}"
image_loaders = build_image_loaders(
    manifest,
    batch_size=IMAGE_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_training=False,
    image_transform=uni_transform,
)
feature_caches = extract_cls_feature_cache(
    uni_model,
    image_loaders,
    DEVICE,
    FEATURE_DIR,
    overwrite=FORCE_FEATURE_REBUILD,
    preprocessing_id=UNI_PREPROCESSING_ID,
)
feature_loaders = build_feature_loaders(
    feature_caches,
    batch_size=FEATURE_BATCH_SIZE,
)
print("UNI feature dimension:", uni_model.feature_dim)
print("UNI data config:", uni_data_config)
print({split: tuple(cache["features"].shape) for split, cache in feature_caches.items()})

## 3. One-seed verification

Train and evaluate seed 41 before enabling the multi-seed section. Only the lightweight head is saved; the large frozen UNI checkpoint is not duplicated.

In [ ]:
RUN_ONE_SEED = True
FORCE_ONE_SEED_RETRAIN = False
one_seed_checkpoint = CHECKPOINT_DIR / f"uni_frozen_head_seed_{SEED}.pt"

if RUN_ONE_SEED and (FORCE_ONE_SEED_RETRAIN or not one_seed_checkpoint.is_file()):
    one_seed_history = fit_frozen_head(
        uni_model,
        feature_loaders["train"],
        feature_loaders["validation"],
        DEVICE,
        one_seed_checkpoint,
        seed=SEED,
        epochs=HEAD_EPOCHS,
        learning_rate=HEAD_LEARNING_RATE,
    )
    one_seed_history.to_csv(ARTIFACT_DIR / f"history_uni_seed_{SEED}.csv", index=False)
elif one_seed_checkpoint.is_file():
    load_classifier_head(uni_model, one_seed_checkpoint, map_location=DEVICE)
else:
    raise FileNotFoundError("Enable RUN_ONE_SEED or provide the seed-41 head checkpoint")

one_seed_evaluation = evaluate_frozen_head(
    uni_model,
    feature_loaders["test"],
    DEVICE,
    CLASS_NAMES,
)
one_seed_metrics = pd.DataFrame([classification_row(one_seed_evaluation, SEED)])
one_seed_metrics.to_csv(ARTIFACT_DIR / "classification_one_seed.csv", index=False)
np.save(ARTIFACT_DIR / f"confusion_uni_seed_{SEED}.npy", one_seed_evaluation["confusion_matrix"])
pd.DataFrame(one_seed_evaluation["classification_report"]).transpose().to_csv(
    ARTIFACT_DIR / f"class_report_uni_seed_{SEED}.csv"
)
prediction_table = pd.DataFrame(
    {
        "path": one_seed_evaluation["paths"],
        "label": one_seed_evaluation["labels"],
        "prediction": one_seed_evaluation["predictions"],
        "confidence": one_seed_evaluation["probabilities"].max(axis=1),
    }
)
prediction_table.to_csv(ARTIFACT_DIR / f"predictions_uni_seed_{SEED}.csv", index=False)
display(one_seed_metrics.style.format(precision=4))

In [ ]:
sanity_batch = next(iter(image_loaders["test"]))
sanity_image = sanity_batch["image"][:1].to(DEVICE)
with torch.inference_mode():
    sanity_tokens = uni_model.forward_tokens(sanity_image)
assert sanity_tokens.shape[1] == 1 + UNI_GRID_SIZE**2
assert np.isfinite(one_seed_metrics[["accuracy", "balanced_accuracy", "macro_f1"]]).all().all()
print("Token tensor:", tuple(sanity_tokens.shape))
print("Patch grid:", UNI_GRID_SIZE, "x", UNI_GRID_SIZE)
print("One-seed verification passed")

In [ ]:
figure, axis = plt.subplots(figsize=(7, 6))
sns.heatmap(
    one_seed_evaluation["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=axis,
)
axis.set(title=f"Frozen UNI classifier, seed {SEED}", xlabel="Predicted", ylabel="True")
axis.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.savefig(PLOT_DIR / f"confusion_uni_seed_{SEED}.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. Preserve classification metrics separately

The table below reads earlier baseline results without modifying them. Attribution faithfulness metrics are written to different files and should not be interpreted as classification metrics.

In [ ]:
classification_tables = [one_seed_metrics.assign(source="uni_attribution")]
for candidate in (
    BASELINE_ARTIFACT_DIR / "test_summary_reverse_mask.csv",
    BASELINE_ARTIFACT_DIR / "test_summary.csv",
):
    if candidate.is_file():
        previous_metrics = pd.read_csv(candidate)
        previous_metrics["source"] = candidate.name
        classification_tables.append(previous_metrics)
        break
classification_comparison = pd.concat(classification_tables, ignore_index=True, sort=False)
classification_comparison.to_csv(ARTIFACT_DIR / "classification_comparison.csv", index=False)
display(classification_comparison.style.format(precision=4))

## 5. Optional full patch-token cache

`forward_features()` returns 196 patch tokens plus CLS. The complete float16 patch-token cache is approximately 2 GB, so it is implemented but disabled until the one-seed run and available disk space are verified.

In [ ]:
RUN_PATCH_TOKEN_EXTRACTION = False
if RUN_PATCH_TOKEN_EXTRACTION:
    all_images = UNIImageDataset(
        manifest,
        augment=False,
        image_transform=uni_transform,
    )
    all_loader = DataLoader(
        all_images,
        batch_size=IMAGE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )
    token_path, token_index_path = cache_patch_tokens(
        uni_model,
        all_loader,
        TOKEN_DIR / "uni_patch_tokens_timm_float16.npy",
        TOKEN_DIR / "uni_patch_token_index_timm.csv",
        DEVICE,
        overwrite=False,
    )
    print("Patch tokens:", token_path)
    print("Token index:", token_index_path)

## 6. UNI attribution and primary faithfulness evaluation

For representative correct and incorrect predictions, compute final-layer CLS attention, attention rollout, predicted-class gradient-weighted attention rollout, and one-patch-at-a-time occlusion. Each attribution is evaluated with top-, random-, and lowest-ranked patch deletion. Occluded regions are set to the normalized ImageNet mean; the controls help reveal generic perturbation damage, but they do not eliminate all out-of-distribution effects.

In [ ]:
RUN_UNI_ATTRIBUTION = True
representative_examples = select_representative_examples(
    one_seed_evaluation,
    correct_count=4,
    incorrect_count=4,
)
representative_examples.to_csv(ARTIFACT_DIR / "representative_examples.csv", index=False)

if RUN_UNI_ATTRIBUTION:
    uni_attribution_metrics, uni_deletion_curves = evaluate_uni_examples(
        uni_model,
        representative_examples,
        DEVICE,
        CLASS_NAMES,
        ARTIFACT_DIR,
        random_repeats=20,
        random_seed=SEED,
        image_transform=uni_transform,
    )
    uni_attribution_summary = (
        uni_attribution_metrics.groupby(["correct", "attribution_method"])
        .agg(
            images=("path", "nunique"),
            mean_occlusion_spearman=("attribution_occlusion_spearman", "mean"),
            mean_top_deletion_auc=("top_deletion_auc", "mean"),
            mean_random_deletion_auc=("random_deletion_auc", "mean"),
            mean_low_deletion_auc=("low_deletion_auc", "mean"),
        )
        .reset_index()
    )
    uni_attribution_summary.to_csv(
        ARTIFACT_DIR / "uni_attribution_summary.csv", index=False
    )
    display(uni_attribution_summary.style.format(precision=4))

In [ ]:
if RUN_UNI_ATTRIBUTION:
    methods = list(uni_deletion_curves["method"].unique())
    figure, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4), sharey=True)
    if len(methods) == 1:
        axes = [axes]
    for axis, method in zip(axes, methods):
        method_curves = uni_deletion_curves[uni_deletion_curves["method"] == method]
        sns.lineplot(
            data=method_curves,
            x="fraction_removed",
            y="normalized_probability",
            hue="strategy",
            marker="o",
            errorbar="se",
            ax=axis,
        )
        axis.set(title=method, xlabel="Fraction of patches removed", ylabel="Normalized target probability")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "uni_deletion_curves.png", dpi=180, bbox_inches="tight")
    plt.show()

## 7. Compare against CNN Grad-CAM

The existing seed-41 RGB CNN is loaded without retraining. Grad-CAM is evaluated against patch occlusion and deletion on the same representative image paths and a normalized 14 x 14 spatial grid.

In [ ]:
RUN_CNN_COMPARISON = True
cnn_checkpoint_candidates = [
    BASELINE_ARTIFACT_DIR / "checkpoints" / "multiseed" / f"rgb_seed_{SEED}.pt",
    BASELINE_ARTIFACT_DIR / "checkpoints" / "baseline_rgb.pt",
]
cnn_checkpoint = next((path for path in cnn_checkpoint_candidates if path.is_file()), None)

if RUN_CNN_COMPARISON and cnn_checkpoint is not None:
    cnn_model = BaselineCNN(in_channels=3, num_classes=len(CLASS_NAMES)).to(DEVICE)
    cnn_state = torch.load(cnn_checkpoint, map_location=DEVICE)
    if isinstance(cnn_state, dict) and "model_state_dict" in cnn_state:
        cnn_state = cnn_state["model_state_dict"]
    cnn_model.load_state_dict(cnn_state)
    cnn_model.eval()
    cnn_target_layer = cnn_model.features[-1][3]
    cnn_attribution_metrics, cnn_deletion_curves = evaluate_cnn_examples(
        cnn_model,
        cnn_target_layer,
        representative_examples,
        DEVICE,
        CLASS_NAMES,
        ARTIFACT_DIR,
        random_repeats=20,
        random_seed=SEED,
    )
    interpretability_comparison = pd.concat(
        (uni_attribution_metrics, cnn_attribution_metrics),
        ignore_index=True,
    )
    interpretability_comparison.to_csv(
        ARTIFACT_DIR / "interpretability_model_comparison.csv", index=False
    )
    interpretability_summary = (
        interpretability_comparison.groupby(["model", "attribution_method"])
        .agg(
            mean_occlusion_spearman=("attribution_occlusion_spearman", "mean"),
            mean_top_deletion_auc=("top_deletion_auc", "mean"),
            mean_random_deletion_auc=("random_deletion_auc", "mean"),
            mean_low_deletion_auc=("low_deletion_auc", "mean"),
        )
        .reset_index()
    )
    interpretability_summary.to_csv(
        ARTIFACT_DIR / "interpretability_model_summary.csv", index=False
    )
    display(interpretability_summary.style.format(precision=4))
elif RUN_CNN_COMPARISON:
    print("CNN checkpoint not found. Expected one of:")
    for path in cnn_checkpoint_candidates:
        print(" -", path)

## 8. Multi-seed head training and explanation stability

Enable this section only after the one-seed classification and attribution checks pass. The frozen UNI features are reused, and each seed saves only its classifier head. Stability is measured for predicted-class gradient-weighted rollout; raw attention and ordinary rollout are encoder-only and therefore identical across head-training seeds.

In [ ]:
RUN_MULTI_SEED = True
multi_seed_path = ARTIFACT_DIR / "classification_multiseed.csv"
if multi_seed_path.is_file():
    multi_seed_results = pd.read_csv(multi_seed_path)
else:
    multi_seed_results = pd.DataFrame()

if RUN_MULTI_SEED:
    rows = [] if multi_seed_results.empty else multi_seed_results.to_dict("records")
    completed_seeds = set() if multi_seed_results.empty else set(multi_seed_results["seed"])
    for training_seed in MULTI_SEEDS:
        checkpoint_path = CHECKPOINT_DIR / f"uni_frozen_head_seed_{training_seed}.pt"
        if checkpoint_path.is_file():
            load_classifier_head(uni_model, checkpoint_path, map_location=DEVICE)
        else:
            history = fit_frozen_head(
                uni_model,
                feature_loaders["train"],
                feature_loaders["validation"],
                DEVICE,
                checkpoint_path,
                seed=training_seed,
                epochs=HEAD_EPOCHS,
                learning_rate=HEAD_LEARNING_RATE,
            )
            history.to_csv(ARTIFACT_DIR / f"history_uni_seed_{training_seed}.csv", index=False)
        if training_seed not in completed_seeds:
            evaluation = evaluate_frozen_head(
                uni_model,
                feature_loaders["test"],
                DEVICE,
                CLASS_NAMES,
            )
            rows.append(classification_row(evaluation, training_seed))
            multi_seed_results = pd.DataFrame(rows).drop_duplicates("seed", keep="last").sort_values("seed")
            multi_seed_results.to_csv(multi_seed_path, index=False)
            completed_seeds.add(training_seed)
    display(
        multi_seed_results[["accuracy", "balanced_accuracy", "macro_f1", "test_loss"]]
        .agg(["mean", "std"])
        .style.format(precision=4)
    )

In [ ]:
RUN_ATTRIBUTION_STABILITY = True
if RUN_ATTRIBUTION_STABILITY:
    checkpoints_by_seed = {
        seed: CHECKPOINT_DIR / f"uni_frozen_head_seed_{seed}.pt"
        for seed in MULTI_SEEDS
        if (CHECKPOINT_DIR / f"uni_frozen_head_seed_{seed}.pt").is_file()
    }
    if len(checkpoints_by_seed) < 2:
        raise RuntimeError("Train at least two UNI heads before stability analysis")

    stability_rows = []
    prediction_frames = []
    for example_index, row in representative_examples.head(4).iterrows():
        image = load_normalized_image(
            row["path"],
            224,
            DEVICE,
            image_transform=uni_transform,
        )
        seed_predictions, stability = evaluate_uni_seed_stability(
            uni_model,
            image,
            checkpoints_by_seed,
            DEVICE,
        )
        seed_predictions.insert(0, "path", row["path"])
        prediction_frames.append(seed_predictions)
        stability_rows.append({"path": row["path"], **stability})

    stability_results = pd.DataFrame(stability_rows)
    stability_predictions = pd.concat(prediction_frames, ignore_index=True)
    stability_results.to_csv(ARTIFACT_DIR / "attribution_stability.csv", index=False)
    stability_predictions.to_csv(
        ARTIFACT_DIR / "attribution_stability_predictions.csv", index=False
    )
    display(stability_results.style.format(precision=4))

## Interpretation checklist

- A faithful map should have positive attribution-occlusion rank correlation.
- Removing highest-attribution patches should reduce the target score faster than random or lowest-attribution deletion; correspondingly, top-deletion AUC should be lower.
- Agreement among raw attention, rollout, and gradient attribution is descriptive, not proof of faithfulness.
- Correct and incorrect predictions should be shown separately.
- Classification accuracy and attribution faithfulness answer different questions and must remain separate in the poster or manuscript.
- Describe highlighted patches only as regions contributing to the model prediction unless independent pathological annotations become available.